<a href="https://colab.research.google.com/github/bigwisu/citrus/blob/main/citrus_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🍊 Install & Import CITRUS
!pip install -q ibm-watsonx-ai openai kneed sqlite-vec matplotlib seaborn
!wget -q -O citrus_engine.py https://raw.githubusercontent.com/bigwisu/citrus/main/citrus_engine.py

from citrus_engine import CitrusEngine
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ CITRUS Engine Loaded.")

In [ ]:
# @title 📂 Step 1: Upload Scopus Data
# @markdown Upload one or more CSV files exported from Scopus.
# @markdown <br> *Note: The system will automatically prepare the 'Title' and 'Abstract' for vectorization.*

import os
from google.colab import files
from tqdm.notebook import tqdm
import pandas as pd

# --- AUTO-INITIALIZE ENGINE (The Fix) ---
# This ensures 'citrus' exists even if you haven't entered API keys yet
if 'CitrusEngine' not in locals():
    try:
        from citrus_engine import CitrusEngine
    except ImportError:
        # Download engine if missing (Fresh session)
        !wget -q -O citrus_engine.py https://raw.githubusercontent.com/bigwisu/citrus/main/citrus_engine.py
        from citrus_engine import CitrusEngine

if 'citrus' not in locals():
    # Initialize a "Dummy" engine just for data processing
    # We will re-initialize with real keys in Step 2
    citrus = CitrusEngine(provider="openai", api_key="placeholder")

# --- UPLOAD LOGIC ---
print("waiting for file upload...")
uploaded = files.upload()

if uploaded:
    file_list = list(uploaded.keys())

    # 2. Pass to Engine for Processing
    with tqdm(total=100, desc="⚙️ Engine Processing", bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt}') as pbar:
        pbar.update(10)

        # CALL THE ENGINE
        df_raw, is_large = citrus.process_scopus_files(file_list)

        pbar.update(90)

    # 3. Output Stats
    print("-" * 40)
    print(f"✅ Ingestion Complete.")
    print(f"📚 Total Documents: {len(df_raw):,}")
    print(f"💾 Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    if is_large:
        print("\n⚠️ **HIGH VOLUME WARNING** ⚠️")
        print(f"You have uploaded {len(df_raw):,} documents (>15,000).")
        print("Recommendation: Use Gemini (Batch Size 100) or IBM Granite.")
    else:
        print("\nNote: Dataset size is optimal.")

    print("-" * 40)

    # Preview
    print("Preview of data prepared for embedding:")
    display(df_raw[['Title', 'text_to_embed']].head(3))

else:
    print("❌ No files uploaded. Please try again.")

In [ ]:
# @title 🧬 Step 2: Vectorize & Build Database (Gemini Optimized)
# @markdown This step creates embeddings using **Google Gemini**.
# @markdown <br> *Note: Gemini Free Tier is limited to 15 requests per minute. The system will auto-throttle.*

# --- 1. CONFIGURATION ---
PROVIDER = "gemini" # @param ["gemini", "ibm"]
API_KEY = "" # @param {type:"string"}
PROJECT_ID = "" # @param {type:"string"} (Only for IBM)
IBM_URL = "https://us-south.ml.cloud.ibm.com" # @param {type:"string"} (Only for IBM)

# @markdown **Performance Tuning**
# Gemini supports up to 100 docs per batch. Maxing this out is CRITICAL for speed.
BATCH_SIZE = 100 # @param {type:"slider", min:10, max:100, step:10}

# --- 2. INITIALIZATION ---
if 'df_raw' not in locals():
    print("❌ Error: Please run Step 1 (Upload) first.")
else:
    # Initialize Engine
    citrus = CitrusEngine(
        provider=PROVIDER,
        api_key=API_KEY,
        project_id=PROJECT_ID,
        ibm_url=IBM_URL
    )

    # Initialize Database
    DB_NAME = "citrus_embeddings.sqlite"
    citrus.init_database(DB_NAME)

    # --- 3. BATCH PROCESSING ---
    import time
    from tqdm.notebook import tqdm

    # SAFETY LOGIC
    if PROVIDER == "gemini":
        # Limit: 15 Requests Per Minute = 1 request every 4 seconds.
        # We set sleep to 4.5s to be safe.
        sleep_time = 4.5
        print(f"🐢 GEMINI MODE: Throttling to 15 RPM (Sleep {sleep_time}s).")
        print(f"   - To go faster: Maximize BATCH_SIZE (Current: {BATCH_SIZE})")
    else:
        sleep_time = 0.1
        print("🐇 IBM MODE: Full Speed.")

    total_docs = len(df_raw)

    with tqdm(total=total_docs, desc="Vectorizing", unit="docs") as pbar:
        for i in range(0, total_docs, BATCH_SIZE):
            batch_df = df_raw.iloc[i : i + BATCH_SIZE]

            # Prepare Text (Ensure they are strings)
            texts = batch_df['text_to_embed'].astype(str).tolist()
            metadatas = batch_df[['DOI', 'Title', 'Year', 'Abstract', 'Authors', 'Source title']].to_dict('records')

            try:
                # Call API
                vectors = citrus.get_embedding(texts)

                # Save
                citrus.save_batch(texts, metadatas, vectors)

                # Update UI
                pbar.update(len(batch_df))

                # Throttle
                time.sleep(sleep_time)

            except Exception as e:
                print(f"\n❌ Error at batch {i}: {e}")
                print("⚠️ Stopping to preserve partial progress.")
                break

    print("-" * 40)
    print(f"✅ Database Ready: {DB_NAME}")

In [ ]:
# @title 🔍 Step 3: Define & Validate Semantic Clusters
# @markdown Enter up to 5 search queries. Leave fields empty if not needed.
# @markdown <br> *The system will calculate the Jaccard Overlap to ensure your topics are distinct.*

# --- 1. CLUSTER INPUTS ---
# We use separate fields for Label (Short name) and Query (Long text)
C1_Label = "Qualitative" # @param {type:"string"}
C1_Query = "Methodologies for automated qualitative coding and thematic analysis of interview transcripts using large language models" # @param {type:"string"}
# @markdown

C2_Label = "Software" # @param {type:"string"}
C2_Query = "Automated generation of executable software code and programming scripts using GitHub Copilot and LLMs" # @param {type:"string"}
# @markdown

C3_Label = "Clinical" # @param {type:"string"}
C3_Query = "Automated classification of clinical records, medical coding, and bibliometric citation analysis" # @param {type:"string"}
# @markdown

C4_Label = "Reliability" # @param {type:"string"}
C4_Query = "Benchmarking inter-rater reliability and agreement consistency between human coders and generative AI" # @param {type:"string"}
# @markdown

C5_Label = "Theory" # @param {type:"string"}
C5_Query = "Epistemological implications and biases of using artificial intelligence in qualitative data analysis" # @param {type:"string"}

# --- 2. PREPARE DATA ---
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate into a clean dictionary, ignoring empty slots
raw_inputs = [
    (C1_Label, C1_Query), (C2_Label, C2_Query),
    (C3_Label, C3_Query), (C4_Label, C4_Query), (C5_Label, C5_Query)
]

active_clusters = {lbl: qry for lbl, qry in raw_inputs if lbl.strip() and qry.strip()}

if len(active_clusters) < 2:
    print("⚠️ Please define at least 2 clusters to perform an overlap check.")
else:
    print(f"⚙️ Analyzing Orthogonality for {len(active_clusters)} clusters...")

    # --- 3. CALL ENGINE ---
    jaccard_matrix = citrus.compute_jaccard_matrix(active_clusters, top_k=50)

    # --- 4. VISUALIZE ---
    plt.figure(figsize=(8, 6))
    sns.heatmap(jaccard_matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
    plt.title("Cluster Orthogonality Check (Jaccard Index)")
    plt.show()

    # --- 5. AUTOMATED RECOMMENDATIONS ---
    print("\n📋 CITRUS ADVISOR RECOMMENDATIONS:")
    print("-" * 40)

    issues_found = False
    labels = list(active_clusters.keys())

    # Iterate upper triangle only
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            l1, l2 = labels[i], labels[j]
            score = jaccard_matrix.loc[l1, l2]

            if score > 0.8:
                print(f"🔴 CRITICAL: '{l1}' and '{l2}' have {score:.2f} overlap.")
                print(f"   ACTION: These are likely synonyms. MERGE them into one query.")
                issues_found = True
            elif score > 0.5:
                print(f"🟠 WARNING: '{l1}' and '{l2}' have {score:.2f} overlap.")
                print(f"   ACTION: Ensure the distinction is intentional.")
                issues_found = True

    if not issues_found:
        print("✅ EXCELLENT: All clusters are semantically distinct (Orthogonal).")
        print("   You may proceed to HiLAT Truncation.")

In [ ]:
# @title 📉 Step 4: Interactive HiLAT Calibration (The Slider)
# @markdown **Instructions:**
# @markdown 1. Select a Cluster.
# @markdown 2. Wait for the "Knee" suggestion (Red Line).
# @markdown 3. **Drag the slider** to adjust the cut-off. Watch the "Border Inspection" tables change.
# @markdown 4. Click **"CONFIRM CUT-OFF"** to save your decision for that cluster.

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import pandas as pd

# Global storage for your decisions
if 'cluster_cutoffs' not in locals():
    cluster_cutoffs = {k: 0 for k in active_clusters.keys()}

# --- WIDGET DEFINITIONS ---
style = {'description_width': 'initial'}

w_cluster = widgets.Dropdown(
    options=list(active_clusters.keys()),
    description='Select Cluster:',
    style=style
)

w_slider = widgets.IntSlider(
    value=10, min=1, max=200, step=1,
    description='Cut-off Rank (n):',
    continuous_update=False, # Update only on release to save computation
    layout=widgets.Layout(width='80%'),
    style=style
)

w_confirm_btn = widgets.Button(
    description="✅ CONFIRM CUT-OFF",
    button_style='success', # Green
    layout=widgets.Layout(width='30%'),
    icon='check'
)

w_output_graph = widgets.Output()
w_output_audit = widgets.Output()
w_output_log = widgets.Output()

# --- STATE ---
current_df = None # Holds the search results for the active cluster

# --- LOGIC ---

def on_cluster_change(change):
    global current_df
    label = change['new']
    query = active_clusters[label]

    with w_output_graph:
        clear_output(wait=True)
        print(f"🔄 Vectorizing & Searching for '{label}'...")

        # 1. Search
        vec = citrus.get_embedding(query)
        current_df = citrus.search_similarity(vec, limit=300)

        # 2. Find Math Knee
        knee, _ = citrus.calculate_cliff(current_df['similarity'].tolist())

        # 3. Update Slider Default
        w_slider.value = knee

        # Trigger redraw
        update_view(knee)

def update_view(cutoff):
    if current_df is None: return

    # --- A. DRAW GRAPH ---
    with w_output_graph:
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(current_df.index, current_df['similarity'], label='Decay Curve', color='#333333')
        plt.axvline(cutoff, color='red', linestyle='--', linewidth=2, label=f'Current Cut-off (n={cutoff})')

        # Add context (Knee)
        knee, _ = citrus.calculate_cliff(current_df['similarity'].tolist())
        if cutoff != knee:
            plt.axvline(knee, color='blue', linestyle=':', alpha=0.5, label=f'Algo Suggestion (n={knee})')

        plt.title(f"Semantic Decay: {w_cluster.value}")
        plt.xlabel("Rank")
        plt.ylabel("Similarity")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    # --- B. DRAW BORDER INSPECTION ---
    with w_output_audit:
        clear_output(wait=True)

        # Pandas formatting to show full text
        pd.set_option('display.max_colwidth', None)

        # Get "Keep Zone" (Bottom 3 included)
        start_keep = max(0, cutoff - 3)
        keep_df = current_df.iloc[start_keep:cutoff][['title', 'similarity']].copy()

        # Get "Drop Zone" (Top 3 excluded)
        drop_df = current_df.iloc[cutoff:cutoff+3][['title', 'similarity']].copy()

        display(HTML(f"<b>Scanning the Border at Rank {cutoff}:</b>"))

        display(HTML(f"<div style='border-left: 5px solid green; padding-left: 10px; margin-bottom: 10px'><b>✅ LAST INCLUDED PAPERS (The Tail):</b></div>"))
        display(keep_df)

        display(HTML(f"<div style='border-left: 5px solid red; padding-left: 10px;'><b>❌ FIRST EXCLUDED PAPERS (The Drop):</b></div>"))
        display(drop_df)

def on_slider_change(change):
    update_view(change['new'])

def on_confirm(b):
    label = w_cluster.value
    cutoff = w_slider.value
    cluster_cutoffs[label] = cutoff

    # Update the Log Area, not the global output
    with w_output_log:
        print(f"📌 {label}: Cut-off set at n={cutoff}")

    # Visual Feedback on button
    original_text = b.description
    b.description = "SAVED!"
    b.button_style = 'info' # Blue
    import time
    time.sleep(0.5)
    b.description = original_text
    b.button_style = 'success'

# --- BINDINGS ---
w_cluster.observe(on_cluster_change, names='value')
w_slider.observe(on_slider_change, names='value')
w_confirm_btn.on_click(on_confirm)

# --- LAYOUT ---
ui = widgets.VBox([
    widgets.HBox([w_cluster, w_confirm_btn]),
    w_slider,
    w_output_graph,
    w_output_audit,
    widgets.HTML("<b>Decision Log:</b>"),
    w_output_log # Add log at bottom
])

# Initialize First View
on_cluster_change({'new': w_cluster.value})

display(ui)

In [ ]:
# @title 📥 Step 5: Harvest & Download
# @markdown Aggregates the selected papers from all clusters, removes duplicates, and generates the final SOTA dataset.

if 'cluster_cutoffs' not in locals():
    # Initialize a dictionary to store user decisions from Step 4
    cluster_cutoffs = {}

# ... (In Step 4, when user clicks "Confirm", update cluster_cutoffs[label] = knee_idx) ...

# --- RUN HARVEST ---
print("🚜 Harvesting papers based on your HiLAT decisions...")

# We need to pass the query text for re-embedding (or cache it)
# For simplicity, we assume 'citrus' engine has a helper or we pass it
# (This part requires you to ensure 'active_clusters' from Step 3 is available)

final_df, audit_log = citrus.harvest_papers(cluster_cutoffs, active_clusters)

# --- DISPLAY STATS ---
print(audit_log)

# --- PREVIEW ---
print("\nTop 5 Final Candidates:")
display(final_df[['similarity', 'Cluster_Source', 'title', 'year']].head(5))

# --- DOWNLOAD BUTTONS ---
from google.colab import files

final_df.to_csv("citrus_sota_candidates.csv", index=False)
with open("citrus_audit_log.txt", "w") as f:
    f.write(audit_log)

print("\n⬇️ Downloading files...")
files.download("citrus_sota_candidates.csv")
files.download("citrus_audit_log.txt")